eid data source: https://www.pxweb.bfs.admin.ch/pxweb/fr/px-x-1703030000_100/-/px-x-1703030000_100.px/table/tableViewLayout2/ $\newline$
2025 population data source: https://www.pxweb.bfs.admin.ch/pxweb/fr/px-x-0102020000_202/-/px-x-0102020000_202.px/
$\newline$
Note: this data analysis was done without any help from AI.


In [50]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import statsmodels.api as sm

In [256]:
cantons_df = pd.read_csv("cantons_data.csv")
eid_df = pd.read_csv("eid_votation.csv")

In [ ]:
cantons_df = cantons_df[cantons_df["Sex"] != "Sexe - total"]
cantons_df = cantons_df[cantons_df["Age"] != "100 ans ou plus"]
cantons_df = cantons_df[cantons_df["Canton"] != "Suisse"]
cantons_df = cantons_df[cantons_df["Canton"] != "Sans indication"]
col = ["Canton", "Age", "Sex", "Count", "Immigration", "Swiss citizenship acquisition"]
cantons_df = cantons_df[col]

cantons_df.head()

,Canton,Age,Sex,Count,Immigration,Swiss citizenship acquisition
332,Z�rich,18 ans,Homme,6227,29,106
333,Z�rich,19 ans,Homme,5993,34,74
334,Z�rich,20 ans,Homme,6065,39,63
335,Z�rich,21 ans,Homme,5925,32,42
336,Z�rich,22 ans,Homme,5819,38,28


In [258]:
cantons_df["Canton"].unique()

array(['Z�rich', 'Bern / Berne', 'Luzern', 'Uri', 'Schwyz', 'Obwalden',
       'Nidwalden', 'Glarus', 'Zug', 'Fribourg / Freiburg', 'Solothurn',
       'Basel-Stadt', 'Basel-Landschaft', 'Schaffhausen',
       'Appenzell Ausserrhoden', 'Appenzell Innerrhoden', 'St. Gallen',
       'Graub�nden / Grigioni / Grischun', 'Aargau', 'Thurgau', 'Ticino',
       'Vaud', 'Valais / Wallis', 'Neuch�tel', 'Gen�ve', 'Jura'],
      dtype=object)

In [259]:
def fix_cantons(cantons_df):
    cantons_df = cantons_df.replace('Z�rich','Zurich')
    cantons_df = cantons_df.replace('Bern / Berne','Berne')
    cantons_df = cantons_df.replace('Fribourg / Freiburg','Fribourg')
    cantons_df = cantons_df.replace('Graub�nden / Grigioni / Grischun','Grischun')
    cantons_df = cantons_df.replace('Valais / Wallis','Valais')
    cantons_df = cantons_df.replace('Neuch�tel','Neuchatel')
    cantons_df = cantons_df.replace('Gen�ve','Geneve')

    return cantons_df

cantons_df = fix_cantons(cantons_df)
eid_df = fix_cantons(eid_df)



In [260]:
cantons_df["Canton"].unique()

array(['Zurich', 'Berne', 'Luzern', 'Uri', 'Schwyz', 'Obwalden',
       'Nidwalden', 'Glarus', 'Zug', 'Fribourg', 'Solothurn',
       'Basel-Stadt', 'Basel-Landschaft', 'Schaffhausen',
       'Appenzell Ausserrhoden', 'Appenzell Innerrhoden', 'St. Gallen',
       'Grischun', 'Aargau', 'Thurgau', 'Ticino', 'Vaud', 'Valais',
       'Neuchatel', 'Geneve', 'Jura'], dtype=object)

In [261]:
eid_df.head()

,Canton,Voter registrations,Vote count,Participation %,Valid votes,Yes,No,Yes %
0,Suisse,5641040,2796897,49.58,2747948,1384586,1363362,50.39
1,Zurich,976490,509297,52.16,503923,274702,229221,54.51
2,Berne,750745,358855,47.80,354158,171771,182387,48.50
3,Luzern,288363,153604,53.27,151603,80261,71342,52.94
4,Uri,27226,12731,46.76,12567,5109,7458,40.65


In [262]:
cantons_df.head()

,Canton,Age,Sex,Count,Immigration,Swiss citizenship acquisition
332,Zurich,18 ans,Homme,6227,29,106
333,Zurich,19 ans,Homme,5993,34,74
334,Zurich,20 ans,Homme,6065,39,63
335,Zurich,21 ans,Homme,5925,32,42
336,Zurich,22 ans,Homme,5819,38,28


In [263]:
age_groups = [range(18, 24), range(24,30), range(30, 36), range(36, 42), range(42, 48), range(48, 54), range(54, 60), range(60, 66),
              range(66, 72), range(72, 78), range(78, 84), range(84, 90), range(90, 96)]
def age_sort(df, age_groups):
    for i in range(95, 100):
        df = df[df["Age"] != str(i) + " ans"]
    # group by ages to count population groups
    for age_group in age_groups:
        new_string = str(age_group[0]) + " - " + str(age_group[-1])
        for age in age_group:
            age_string = str(age) + " ans"
            df = df.replace(age_string, new_string)
    df.reset_index()
    return df
        

In [264]:
# represent demographics with percentages instead of count
def canton_proportions(df):
    df["Count"] = df["Count"] / df.groupby("Canton")["Count"].transform('sum')
    return df

In [265]:
cantons_df = canton_proportions(cantons_df)
cantons_df.head()

,Canton,Age,Sex,Count,Immigration,Swiss citizenship acquisition
332,Zurich,18 ans,Homme,0.006559,29,106
333,Zurich,19 ans,Homme,0.006313,34,74
334,Zurich,20 ans,Homme,0.006389,39,63
335,Zurich,21 ans,Homme,0.006241,32,42
336,Zurich,22 ans,Homme,0.006129,38,28


In [266]:
cantons_df = age_sort(cantons_df, age_groups)
cantons_df = cantons_df.groupby(["Canton", "Age", "Sex"]).sum()
cantons_df.head(10)

Count  Immigration  Swiss citizenship acquisition
Canton Age     Sex                                                        
Aargau 18 - 23 Femme  0.035195           53                            130
               Homme  0.037526           52                            123
       24 - 29 Femme  0.037362           76                             89
               Homme  0.039086           71                             37
       30 - 35 Femme  0.042295           57                            123
               Homme  0.041897           79                             88
       36 - 41 Femme  0.044499           50                            224
               Homme  0.044360           57                            213
       42 - 47 Femme  0.045031           35                            228
               Homme  0.044403           50                            208

In [ ]:
cantons_df = canton_proportions(cantons_df)

In [257]:
eid_df = eid_df.groupby(["Canton"]).sum()
eid_df.head()

,Voter registrations,Vote count,Participation %,Valid votes,Yes,No,Yes %
Canton,,,,,,,
Aargau,447798,232292,51.87,230470,113121,117349,49.08
Appenzell Ausserrhoden,39526,21896,55.40,21652,9949,11703,45.95
Appenzell Innerrhoden,12385,6006,48.49,5859,2644,3215,45.13
Basel-Landschaft,192244,96303,50.09,93940,46245,47695,49.23
Basel-Stadt,114378,55409,48.44,54544,30976,23568,56.79


In [ ]:
df = pd.DataFrame({
    "Canton": cantons_df["Canton"].unique
    
})

In [ ]:


smf.ols(formula='Head_size ~ Brain_weight', data=df).fit()